In [1]:
#Spark connection with S3 options
import os
import socket
from pyspark.sql import SparkSession

# Замените следующие значения на свои credentials
aws_access_key = "j612yzOjcIwYyPhVD14P"
aws_secret_key = "bVTEwJ7sJ0TboubM5wJcn2ieYrV91uXk3N9dJ457"
s3_bucket = "startde-datasets"
s3_endpoint_url = "https://s3.lab.karpov.courses"
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = """/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar, 
/nfs/env/lib/python3.8/site-packages/pyspark/jars/hadoop-aws-3.3.4.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/aws-java-sdk-bundle-1.12.433.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/postgresql-42.7.4.jar
"""

MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        config("spark.hadoop.fs.s3a.access.key", aws_access_key). \
        config("spark.hadoop.fs.s3a.secret.key", aws_secret_key). \
        config("fs.s3a.endpoint", s3_endpoint_url).  \
        config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem"). \
        config("spark.hadoop.fs.s3a.committer.name", "directory"). \
        config("spark.hadoop.fs.s3a.path.style.access", True). \
        config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"). \
        config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false"). \
        getOrCreate()

26/02/08 09:50:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Read sellers table
sellers_table = spark.read.parquet("s3a://startde-datasets/shared/sellers.parquet")

In [15]:
sellers_table.show()

+---------+-----------+------------+
|seller_id|seller_name|daily_target|
+---------+-----------+------------+
|        0|   seller_0|        2500|
|        1|   seller_1|       16451|
|        2|   seller_2|        2855|
|        3|   seller_3|       19103|
|        4|   seller_4|        8820|
|        5|   seller_5|       14894|
|        6|   seller_6|        7928|
|        7|   seller_7|       17022|
|        8|   seller_8|       19924|
|        9|   seller_9|        6496|
+---------+-----------+------------+



+---------+-----------+------------+
|seller_id|seller_name|daily_target|
+---------+-----------+------------+
|        0|   seller_0|        2500|
|        1|   seller_1|       16451|
|        2|   seller_2|        2855|
|        3|   seller_3|       19103|
|        4|   seller_4|        8820|
|        5|   seller_5|       14894|
|        6|   seller_6|        7928|
|        7|   seller_7|       17022|
|        8|   seller_8|       19924|
|        9|   seller_9|        6496|
+---------+-----------+------------+



In [21]:
overall_target = sellers_table.agg(sum(col("daily_target"))).collect()[0][0]
print(overall_target)
sellers_table.show()

115993
+---------+-----------+------------+
|seller_id|seller_name|daily_target|
+---------+-----------+------------+
|        0|   seller_0|        2500|
|        1|   seller_1|       16451|
|        2|   seller_2|        2855|
|        3|   seller_3|       19103|
|        4|   seller_4|        8820|
|        5|   seller_5|       14894|
|        6|   seller_6|        7928|
|        7|   seller_7|       17022|
|        8|   seller_8|       19924|
|        9|   seller_9|        6496|
+---------+-----------+------------+



In [35]:
result = sellers_table.withColumn('daily_target_percentage', col('daily_target') / overall_target * 100). \
    orderBy(desc('daily_target')). \
    limit(3)


In [36]:
result.show()

+---------+-----------+------------+-----------------------+
|seller_id|seller_name|daily_target|daily_target_percentage|
+---------+-----------+------------+-----------------------+
|        8|   seller_8|       19924|      17.17689860595036|
|        3|   seller_3|       19103|      16.46909727311131|
|        7|   seller_7|       17022|      14.67502349279698|
+---------+-----------+------------+-----------------------+



In [37]:
pandas_df = result.toPandas()
pandas_df.to_parquet('result.parquet')


In [40]:
spark.stop()